# Project UTS

## Pengisian Missing Value

#### Menggunakan WKNN Manual
Pada tahap awal preprocessing dilakukan pengecekan terhadap ****missing value**** atau nilai yang kosong pada dataset. Missing value dapat menyebabkan proses perhitungan jarak tidak berjalan dengan baik karena setiap atribut harus memiliki nilai.

Oleh karena itu, nilai yang kosong perlu diisi terlebih dahulu. Proses ini disebut ****imputasi data****. Salah satu metode yang dapat digunakan adalah mengisi nilai yang kosong menggunakan ****nilai rata-rata (mean)**** atau nilai yang paling mendekati dari atribut tersebut.

Dengan melakukan pengisian missing value, dataset menjadi lengkap sehingga dapat digunakan pada proses selanjutnya yaitu ****normalisasi data**** dan perhitungan menggunakan metode ****Weighted K-Nearest Neighbor (WKNN)****.

---

#### **Data Asli**

Berikut merupakan dataset sebelum dilakukan proses normalisasi.

| Data | IPK | PO | JML |
|----|----|----|----|
|1|2|2|0|
|2|2|3|1|
|3|4|3|0|
|4|2|2|1|
|5|2|2|0|
|6|4|3|1|

Keterangan:

- ****IPK**** : Indeks Prestasi Kumulatif  
- ****PO**** : Penghasilan Orang Tua  
- ****JML**** : Jumlah tanggungan / kelas target  

---

#### Normalisasi Menggunakan Min-Max

Karena nilai pada setiap atribut memiliki skala yang berbeda, maka dilakukan proses ****normalisasi Min-Max**** agar seluruh nilai berada pada rentang yang sama yaitu ****0 sampai 1****.

Rumus normalisasi Min-Max:

$$
X' = \frac{X - X_{min}}{X_{max} - X_{min}}
$$

Keterangan:

- $X$ = nilai asli  
- $X_{min}$ = nilai minimum  
- $X_{max}$ = nilai maksimum  

Sebagai contoh normalisasi pada atribut ****IPK****:

Nilai minimum IPK = 2  
Nilai maksimum IPK = 4  

$$
\frac{2-2}{4-2}=0
$$

$$
\frac{4-2}{4-2}=1
$$

**Hasil normalisasi data menjadi sebagai berikut:**

| Data | IPK | PO | JML |
|----|----|----|----|
|1|0|0|0|
|2|0|1|1|
|3|1|1|0|
|4|0|0|1|
|5|0|0|0|
|6|1|1|1|

---

#### **Langkah 1: Hitung Jarak dan Kemiripan**

Setelah data dinormalisasi, langkah berikutnya adalah menghitung ****jarak antara data uji dengan data latih**** menggunakan metode ****Euclidean Distance****.

Rumus Euclidean Distance:

$$
d = \sqrt{(x_1-x_2)^2 + (y_1-y_2)^2}
$$

Perhitungan jarak ini dilakukan berdasarkan atribut ****IPK**** dan ****PO****.

Setelah jarak diperoleh, maka dihitung ****bobot (weight)**** menggunakan rumus:

$$
w = \frac{1}{d}
$$

Semakin kecil nilai jarak, maka bobot yang dihasilkan akan semakin besar.

Hasil bobot yang diperoleh:

| Data | Bobot |
|----|----|
|1|2|
|2|2|
|3|0.895|
|4|2|
|5|2|
|6|0.895|

---

#### **Langkah 2: Hitung Estimasi Menggunakan Weighted Average**

Setelah bobot diperoleh, langkah selanjutnya adalah menghitung estimasi menggunakan ****Weighted Average****.

Rumus:

$$
\hat{y} = \frac{\sum (w_i \times y_i)}{\sum w_i}
$$

Keterangan:

- $w_i$ = bobot  
- $y_i$ = nilai target (JML)

---

#### **Penjumlahan Pembilang**

Pembilang diperoleh dari hasil perkalian antara ****bobot dengan nilai JML****.

| Bobot | JML | Bobot × JML |
|----|----|----|
|2|0|0|
|2|1|2|
|0.895|0|0|
|2|1|2|
|2|0|0|
|0.895|1|0.895|

Jumlah pembilang:

$$
0 + 2 + 0 + 2 + 0 + 0.895 = 4.895
$$

---

#### **Penjumlahan Penyebut**

Penyebut merupakan jumlah seluruh bobot.

$$
2 + 2 + 0.895 + 2 + 2 + 0.895 = 9.79
$$

---

#### **Hasil Akhir**

Nilai akhir diperoleh dengan membagi pembilang dengan penyebut.

$$
\frac{4.895}{9.79} = 0.5
$$

Sehingga hasil estimasi menggunakan metode ****Weighted K-Nearest Neighbor (WKNN)**** adalah:

****0.5****

Nilai ini menunjukkan hasil prediksi berdasarkan kedekatan setiap data terhadap data uji dengan mempertimbangkan bobot dari setiap tetangga terdekat.

#### Menggunakan Code WKNN 

In [1]:
import numpy as np
import pandas as pd

# Data asli
data = pd.DataFrame({
    'IPK': [2,2,4,2,2,4],
    'PO': [2,3,3,2,2,3],
    'JML': [0,1,0,1,0,1]
})

# -----------------------------
# Normalisasi Min-Max
# -----------------------------
def minmax(col):
    return (col - col.min()) / (col.max() - col.min())

data['IPK'] = minmax(data['IPK'])
data['PO'] = minmax(data['PO'])

# -----------------------------
# Data uji
# -----------------------------
test = np.array([0.5, 0.5])

# -----------------------------
# Hitung Euclidean Distance
# -----------------------------
distances = []

for i in range(len(data)):
    point = np.array([data.iloc[i]['IPK'], data.iloc[i]['PO']])
    d = np.sqrt(np.sum((point - test)**2))
    distances.append(d)

# -----------------------------
# Hitung Bobot (1/d)
# -----------------------------
weights = [1/d for d in distances]

# -----------------------------
# Weighted Average
# -----------------------------
numerator = 0
denominator = 0

for i in range(len(data)):
    w = weights[i]
    y = data.iloc[i]['JML']
    
    numerator += w * y
    denominator += w

result = numerator / denominator

print("Bobot:", weights)
print("Pembilang:", numerator)
print("Penyebut:", denominator)
print("Hasil Akhir:", result)

Bobot: [np.float64(1.414213562373095), np.float64(1.414213562373095), np.float64(1.414213562373095), np.float64(1.414213562373095), np.float64(1.414213562373095), np.float64(1.414213562373095)]
Pembilang: 4.242640687119285
Penyebut: 8.48528137423857
Hasil Akhir: 0.5
